In [0]:
import re, json, requests, os
from pyspark.sql import functions as F

In [0]:
JOB_NAME          = "mt-method2: per-node training"
EXPERIMENT_PATH   = "/9900-f18a-cake/classifier"                      
CATALOG           = "cb_prod"
SCHEMA            = "comp9300-9900-f18a-cake"
SAVE_MODE         = "nosave"                                         

In [0]:
# default job-cluster spec for each task
NEW_CLUSTER = {
    "spark_version": "16.4.x-scala2.12",
    "node_type_id":  "Standard_E16_v3",  
    "num_workers":   1,
    "spark_conf": { "spark.databricks.cluster.profile": "singleNode" },
    "custom_tags": { "mt-method2": "child4-per-node" }
}
MAX_PARALLEL_TASKS = 4

In [0]:
def safe_name(s: str) -> str:
    s = re.sub(r"[^\w\-]+", "_", s.strip())
    return s[:120] or "node"

In [0]:
CATALOG = "cb_prod"
SCHEMA  = "comp9300-9900-f18a-cake"  
TABLE   = "node_labels"
NODE_TABLE_FQN = f"`{CATALOG}`.`{SCHEMA}`.`{TABLE}`"

nodes = (
    spark.table(NODE_TABLE_FQN)
         .select(F.col("node_id"))
         .where(F.col("node_id").isNotNull() & (F.length(F.trim(F.col("node_id"))) > 0))
         .distinct()
         .toPandas()["node_id"]
         .drop_duplicates()
         .tolist()
)

print(nodes[:10], f"... ({len(nodes)} total)")

In [0]:
def make_task(node: str) -> dict:
    task_name = safe_name(node)
    # One job-cluster per task:
    return {
        "task_key": task_name,
        "description": f"Train Child4 for node '{node}'",
        "new_cluster": NEW_CLUSTER,
        "notebook_task": {
            "notebook_path": CHILD_NB_PATH,
            "base_parameters": {
                "NODE_ID":  node,
                "ONLY_NODE": node,   
                "SAVE_MODE": SAVE_MODE
            }
        },
        "timeout_seconds": 4 * 60 * 60
    }

tasks = [make_task(n) for n in nodes]
print(f"Prepared {len(tasks)} tasks")

In [0]:
BASE = "https://adb-3174997426428886.6.azuredatabricks.net".rstrip("/")
TOKEN = os.getenv("DATABRICKS_TOKEN") or dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
S = requests.Session()
S.headers.update({"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"})

BASE_JOBS = f"{BASE}/api/2.1/jobs"

In [0]:
def _req(method, path, **kw):
    url = f"{BASE_JOBS}/{path.lstrip('/')}"
    r = S.request(method, url, **kw)
    try:
        print(method, url, "->", r.status_code, json.dumps(r.json(), indent=2)[:800])
    except Exception:
        print(method, url, "->", r.status_code, r.text[:800])
    r.raise_for_status()
    return r

In [0]:
def find_job_id_by_name(name: str):
    r = _req("GET", "list")
    for j in r.json().get("jobs", []):
        if j["settings"].get("name") == name:
            return j["job_id"]
    return None

In [0]:
job_settings = {
  "name": JOB_NAME,
  "max_concurrent_runs": 12,
  "format": "MULTI_TASK",
  "queue": {"enabled": True},
  # minimal one task example pointing to your Child4
  "tasks": [{
      "task_key": "HaemMalig",
      "notebook_task": {
          "notebook_path": "/Workspace/9900-f18a-cake/mt-method2/child4",
          "base_parameters": {
              "NODE_ID": "Haematological malignancy",
              "ONLY_NODE": "Haematological malignancy",
              "SAVE_MODE": "nosave"
          }
      },
      # use either job_clusters or existing_cluster_id:
      "existing_cluster_id": "1006-232549-x1y029bo"
  }]
}

In [0]:
import os, requests, json
BASE = "https://adb-3174997426428886.6.azuredatabricks.net"
TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
r = requests.get(f"{BASE}/api/2.0/clusters/list",
                 headers={"Authorization": f"Bearer {TOKEN}"})
r.raise_for_status()
for c in r.json().get("clusters", []):
    print(c["cluster_name"], c["cluster_id"], c["state"])


In [0]:
job_id = find_job_id_by_name(JOB_NAME)
if job_id:
    # replace-all update
    payload = {"job_id": job_id, "new_settings": job_settings}
    _req("POST", "update", json=payload)   # <— /api/2.1/jobs/update
    print(f"Updated job {job_id}: {JOB_NAME}")
else:
    resp = _req("POST", "create", json=job_settings)  # <— /api/2.1/jobs/create
    job_id = resp.json()["job_id"]
    print(f"Created job {job_id}: {JOB_NAME}")